# Linear Regression

A fundamental supervised learning algorithm for predicting **continuous** numeric values. Covers simple and multiple linear regression, gradient descent, evaluation metrics, and practical implementation.

## Simple Linear Regression

Models the relationship between a single feature *x* and target *y* as a straight line:

$$y = mx + b$$

Where:
- $m$ = slope (coefficient / weight)
- $b$ = intercept (bias term)

For multiple features, this generalizes to:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

Or in vector form: $\hat{y} = X\mathbf{w} + b$

## Cost Function: Mean Squared Error (MSE)

The **MSE** measures the average squared difference between predicted and actual values:

$$J(\mathbf{w}) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Why squared? It penalizes larger errors more heavily and makes the function differentiable (nice convex optimization landscape).

## Gradient Descent Intuition

Gradient descent iteratively updates weights to minimize the cost function:

$$w_j := w_j - \alpha \frac{\partial J}{\partial w_j}$$

Where $\alpha$ is the **learning rate**. We move in the direction opposite the gradient (downhill) to reach the minimum.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
print("All libraries imported.")

In [ ]:
# Load the Diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print("Diabetes dataset:")
print(f"  Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"  Features: {feature_names}")
print(f"  Target range: [{y.min():.1f}, {y.max():.1f}] (disease progression)")
print(f"  Target mean: {y.mean():.1f}")

In [ ]:
# Simple Linear Regression — using a single feature (BMI)
bmi_idx = list(feature_names).index('bmi')
X_bmi = X[:, bmi_idx].reshape(-1, 1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_bmi, y, test_size=0.2, random_state=42
)

# Fit
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# Predict
y_pred = lin_reg.predict(X_test)

print(f"Simple Linear Regression (BMI -> disease progression)")
print(f"  Coefficient (slope): {lin_reg.coef_[0]:.4f}")
print(f"  Intercept (bias):    {lin_reg.intercept_:.4f}")
print(f"  Equation: y = {lin_reg.coef_[0]:.4f} * BMI + {lin_reg.intercept_:.4f}")

In [ ]:
# Visualize the regression line
plt.figure(figsize=(10, 6))
plt.scatter(X_train, y_train, alpha=0.5, label='Training data', color='blue')
plt.scatter(X_test, y_test, alpha=0.7, label='Test data', color='green')

# Plot regression line
X_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_line = lin_reg.predict(X_line)
plt.plot(X_line, y_line, color='red', linewidth=2, label='Regression line')

plt.xlabel('BMI (standardized)')
plt.ylabel('Disease progression')
plt.title('Simple Linear Regression: BMI vs Disease Progression')
plt.legend()
plt.show()

In [ ]:
# Evaluation Metrics

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Evaluation Metrics:")
print(f"  MSE  (Mean Squared Error):     {mse:.2f}")
print(f"  RMSE (Root MSE):               {rmse:.2f}")
print(f"  MAE  (Mean Absolute Error):    {mae:.2f}")
print(f"  R²   (Coefficient of Det.):    {r2:.4f}")
print(f"\nInterpretation:")
print(f"  R² = {r2:.2f} means {r2*100:.1f}% of the variance in the target")
print(f"       is explained by BMI alone.")

In [ ]:
# Multiple Linear Regression — using ALL features
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X, y, test_size=0.2, random_state=42
)

multi_reg = LinearRegression()
multi_reg.fit(X_train_multi, y_train_multi)
y_pred_multi = multi_reg.predict(X_test_multi)

mse_multi = mean_squared_error(y_test_multi, y_pred_multi)
r2_multi = r2_score(y_test_multi, y_pred_multi)

print("Multiple Linear Regression (all features):")
print(f"  MSE: {mse_multi:.2f}")
print(f"  RMSE: {np.sqrt(mse_multi):.2f}")
print(f"  R²:  {r2_multi:.4f}")
print(f"\nComparing models:")
print(f"  Simple (BMI only)   R² = {r2:.4f}")
print(f"  Multiple (all vars) R² = {r2_multi:.4f}")

In [ ]:
# Interpreting Coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': multi_reg.coef_,
    'Abs_Coefficient': np.abs(multi_reg.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("Feature Coefficients (sorted by importance):")
print(coef_df.to_string(index=False))
print("\nInterpretation: A unit increase in a feature changes the")
print("target by the coefficient value (holding others constant).")

In [ ]:
# Visualize predicted vs actual
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(y_test_multi, y_pred_multi, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
plt.xlabel('Actual values')
plt.ylabel('Predicted values')
plt.title('Predicted vs Actual (Multiple LR)')

plt.subplot(1, 2, 2)
residuals = y_test_multi - y_pred_multi
plt.scatter(y_pred_multi, residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Predicted values')
plt.ylabel('Residuals')
plt.title('Residual Plot')

plt.tight_layout()
plt.show()

In [ ]:
# Gradient Descent with SGDRegressor
sgd_reg = SGDRegressor(max_iter=1000, tol=1e-3, penalty=None, eta0=0.01, random_state=42)
sgd_reg.fit(X_train_multi, y_train_multi)
y_pred_sgd = sgd_reg.predict(X_test_multi)

print("SGD Regressor (Gradient Descent) results:")
print(f"  Coefficients: {sgd_reg.coef_[:5]}...")
print(f"  Intercept: {sgd_reg.intercept_:.4f}")
print(f"  R² Score: {r2_score(y_test_multi, y_pred_sgd):.4f}")
print(f"  \nNote: Should be similar to OLS solution above.")

## Assumptions of Linear Regression

1. **Linearity**: The relationship between features and target is linear
2. **Independence**: Observations are independent of each other
3. **Homoscedasticity**: Constant variance of residuals across all predictions
4. **Normality**: Residuals are approximately normally distributed (for inference)
5. **No multicollinearity**: Features are not highly correlated with each other

When assumptions are violated, predictions may still be good but inference on coefficients becomes unreliable.

In [ ]:
# Check multicollinearity with correlation matrix
corr_matrix = np.corrcoef(X.T)
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=feature_names, yticklabels=feature_names)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Assumption check: residual distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.hist(residuals, bins=20, edgecolor='black', alpha=0.7)
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Residual Distribution')

plt.subplot(1, 3, 2)
from scipy import stats
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')

plt.subplot(1, 3, 3)
plt.scatter(y_pred_multi, residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Predicted')
plt.ylabel('Residuals')
plt.title('Homoscedasticity Check')

plt.tight_layout()
plt.show()
print("Residuals should be roughly normal and have constant spread.")

In [ ]:
# Interview Problem: Predict house prices with Linear Regression
# Using Boston-style features (California housing as proxy)

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X_h, y_h = housing.data, housing.target

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

scaler_h = StandardScaler()
X_h_train_scaled = scaler_h.fit_transform(X_h_train)
X_h_test_scaled = scaler_h.transform(X_h_test)

lr_housing = LinearRegression()
lr_housing.fit(X_h_train_scaled, y_h_train)
y_h_pred = lr_housing.predict(X_h_test_scaled)

print("House Price Prediction Results:")
print(f"  Features: {housing.feature_names}")
print(f"  R² Score: {r2_score(y_h_test, y_h_pred):.4f}")
print(f"  RMSE: ${np.sqrt(mean_squared_error(y_h_test, y_h_pred))*100000:.0f} (in $100k units)")

In [ ]:
# Interview Problem: Interpret coefficients for business stakeholders
coef_h = pd.DataFrame({
    'Feature': housing.feature_names,
    'Coefficient': lr_housing.coef_
})

print("House Price Model Coefficients:")
print(coef_h.to_string(index=False))
print("\nInterpretation example:")
print(f"  A 1-std increase in MedInc increases price by "
      f"${lr_housing.coef_[0]*100000:.0f} (in $100k units)")
print(f"  A 1-std increase in AveOccup decreases price by "
      f"${abs(lr_housing.coef_[5])*100000:.0f}")

In [ ]:
# Problem: Find the best single feature for predicting diabetes
from sklearn.metrics import r2_score

best_feature = None
best_score = -np.inf

for i, name in enumerate(feature_names):
    X_uni = X[:, i].reshape(-1, 1)
    X_utr, X_ute, y_utr, y_ute = train_test_split(X_uni, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_utr, y_utr)
    score = r2_score(y_ute, model.predict(X_ute))
    if score > best_score:
        best_score = score
        best_feature = name
    print(f"  {name:8s} R² = {score:.4f}")

print(f"\nBest single feature: {best_feature} (R² = {best_score:.4f})")

## Summary

- **Linear Regression** models the relationship between features and a continuous target
- **MSE** is the most common cost function; **RMSE** is interpretable in target units
- **R²** tells us how much variance is explained by the model
- **Gradient Descent** iteratively finds optimal weights
- Multiple features improve predictions over single-feature models
- Always check **assumptions** (linearity, homoscedasticity, normality of residuals)